## Agrégation des $B$-splines par la médiane:

Permet de rendre les résultats plus interprétables pour la visualisation des classifications, etc. ...

In [27]:
#install.packages("tidyverse")

In [28]:
library("caret")
library("corrplot")
library("corrr")
library("CVglasso")
library("dplyr")
library("e1071")
library("fda")
library("ggplot2")
library("glasso")
library("igraph")
library("mclust")
library("nnet")
library("pls")
library("qgraph")
library("randomForest")
library("readr")
library("splines")
library("tidymodels")
library("tidyverse")
library("VGAM")
library("xgboost")

In [29]:
df_new <- read.csv('../../pipeline_2/data/multispec_bsplines_IGT.csv', row.names = 'X')
head(df_new)

,bspline_1_1,bspline_2_1,bspline_3_1,bspline_4_1,bspline_5_1,bspline_6_1,bspline_7_1,bspline_8_1,bspline_9_1,bspline_1_2,⋯,bspline_9_2,bspline_1_3,bspline_2_3,bspline_3_3,bspline_4_3,bspline_5_3,bspline_6_3,bspline_7_3,bspline_8_3,bspline_9_3
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1-2Dichloroethane0,0.042859950,0.31041698,0.51403305,0.62173780,0.03431683,-0.31216163,-0.16177808,-0.61811774,-0.69611517,0.319171851,⋯,0.25465136,0.17421335,-0.29096780,0.45766335,-0.63651666,-0.6101915,-1.0122583,-0.7202593,-1.0490280,-0.8609281
1-2Dichloroethane1,0.006455640,0.39251760,0.82991897,0.26948011,0.54668293,0.24931517,-0.63373810,0.46927073,-0.33824607,-0.015799672,⋯,0.04691594,-0.03380804,0.05718799,-0.09338539,0.14276143,-0.8036310,-0.6387393,-0.6808124,-0.4320695,-0.6110400
1-2Dichloroethane2,0.327938158,-0.26464513,0.48451374,0.02351809,0.04324316,0.11916632,0.08827694,0.02312422,0.02278163,0.354224106,⋯,-0.34282508,0.15004945,-0.23951635,0.32780252,-0.31169849,-0.7677156,-0.7875300,-0.4524011,-0.7705458,-0.3562878
1-chlorodecane0,-0.001266327,0.02122973,-0.01433533,0.03552926,0.01566774,0.01169483,0.06060685,0.07095725,0.02065529,0.007125068,⋯,0.02228604,0.00000000,0.00000000,0.00000000,0.00000000,0.0000000,0.0000000,0.0000000,0.0000000,0.0000000
4-octylphenol0,0.191932261,-0.28559929,0.46884036,-0.46132427,2.32348311,0.61417648,1.78721517,-0.47782508,0.94929057,-0.054622886,⋯,1.79662924,0.08616678,-0.13326610,0.15897355,-0.03092931,0.1156325,0.3229518,0.1170611,0.9125265,0.4000610
4-octylphenol1,0.191932261,-0.28559929,0.46884036,-0.46132427,2.32348311,0.61417648,1.78721517,-0.47782508,0.94929057,0.068498536,⋯,1.13125647,0.04647230,-0.09804601,0.20415663,0.27047965,0.4234347,0.5865024,1.3926614,0.5701989,0.9129696


In [30]:
## ajout col nom complet substance (filtrage plus simple pour groupage par sub via médiane):
df_name <- read.csv("../../../projet/pipeline_2/data/fPCA_score_agg_Default_IGT.csv", row.names = "Repetition")

In [31]:
df_name[30:40, ]

,X,fPC1,fPC2,fPC3,fPC4,sub_full,y
,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<int>
Carbaryl0,29,0.2216584,4.5410169,-0.23979933,8.3694283,Carbaryl,1
Carbaryl1,30,0.6141355,2.8307450,0.07181435,7.0509765,Carbaryl,1
Chlorothanolil0,31,-4.2157692,-6.2185271,-14.11709524,1.7408734,Chlorothalonil,3
Chlorothanolil1,32,-10.8082462,1.4887476,-5.34452726,-5.5619762,Chlorothalonil,3
Chlorothanolil2,33,-8.0111422,8.1199191,-7.45544061,-1.5065235,Chlorothalonil,3
Chlorpyrifos0,34,3.1473240,0.7937227,0.87630972,1.2636626,Chlorpyrifos,1
Chlorpyrifos1,35,5.7751258,-3.3695465,-0.30917356,-2.7363597,Chlorpyrifos,1
Cypermethrine0,36,8.2715743,-12.6919954,4.52968774,-9.5225959,Cypermethrine,1
Cypermethrine1,37,14.6116240,-12.8808631,4.32391859,-4.1544355,Cypermethrine,1


In [32]:
rownames(df_name)[rownames(df_name) == "Chlorothanolil0"] <- "Chlorothalonil0"
rownames(df_name)[rownames(df_name) == "Chlorothanolil1"] <- "Chlorothalonil1"
rownames(df_name)[rownames(df_name) == "Chlorothanolil2"] <- "Chlorothalonil2"

In [33]:
df_name[30:40, ]

,X,fPC1,fPC2,fPC3,fPC4,sub_full,y
,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<int>
Carbaryl0,29,0.2216584,4.5410169,-0.23979933,8.3694283,Carbaryl,1
Carbaryl1,30,0.6141355,2.8307450,0.07181435,7.0509765,Carbaryl,1
Chlorothalonil0,31,-4.2157692,-6.2185271,-14.11709524,1.7408734,Chlorothalonil,3
Chlorothalonil1,32,-10.8082462,1.4887476,-5.34452726,-5.5619762,Chlorothalonil,3
Chlorothalonil2,33,-8.0111422,8.1199191,-7.45544061,-1.5065235,Chlorothalonil,3
Chlorpyrifos0,34,3.1473240,0.7937227,0.87630972,1.2636626,Chlorpyrifos,1
Chlorpyrifos1,35,5.7751258,-3.3695465,-0.30917356,-2.7363597,Chlorpyrifos,1
Cypermethrine0,36,8.2715743,-12.6919954,4.52968774,-9.5225959,Cypermethrine,1
Cypermethrine1,37,14.6116240,-12.8808631,4.32391859,-4.1544355,Cypermethrine,1


In [34]:
df_new$full_name <- df_name$sub_full[match(rownames(df_new), rownames(df_name))]

In [35]:
## sanity check:
df_new

,bspline_1_1,bspline_2_1,bspline_3_1,bspline_4_1,bspline_5_1,bspline_6_1,bspline_7_1,bspline_8_1,bspline_9_1,bspline_1_2,⋯,bspline_1_3,bspline_2_3,bspline_3_3,bspline_4_3,bspline_5_3,bspline_6_3,bspline_7_3,bspline_8_3,bspline_9_3,full_name
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
1-2Dichloroethane0,0.0428599502,0.3104169809,0.5140330488,0.6217378048,0.0343168263,-0.3121616308,-0.1617780782,-0.6181177421,-0.6961151714,0.3191718511,⋯,0.1742133519,-0.290967797,0.457663353,-0.636516656,-0.610191538,-1.012258286,-0.720259341,-1.049027966,-0.860928113,1-2Dichloroethane
1-2Dichloroethane1,0.0064556398,0.3925176002,0.8299189711,0.2694801130,0.5466829256,0.2493151690,-0.6337381047,0.4692707261,-0.3382460723,-0.0157996724,⋯,-0.0338080388,0.057187990,-0.093385392,0.142761434,-0.803631031,-0.638739310,-0.680812425,-0.432069494,-0.611039982,1-2Dichloroethane
1-2Dichloroethane2,0.3279381580,-0.2646451337,0.4845137377,0.0235180885,0.0432431615,0.1191663208,0.0882769398,0.0231242212,0.0227816304,0.3542241057,⋯,0.1500494456,-0.239516349,0.327802521,-0.311698492,-0.767715644,-0.787530033,-0.452401110,-0.770545783,-0.356287780,1-2Dichloroethane
1-chlorodecane0,-0.0012663269,0.0212297344,-0.0143353314,0.0355292603,0.0156677416,0.0116948309,0.0606068544,0.0709572486,0.0206552934,0.0071250680,⋯,0.0000000000,0.000000000,0.000000000,0.000000000,0.000000000,0.000000000,0.000000000,0.000000000,0.000000000,1-chlorodecane
4-octylphenol0,0.1919322615,-0.2855992927,0.4688403629,-0.4613242667,2.3234831092,0.6141764850,1.7872151736,-0.4778250813,0.9492905666,-0.0546228864,⋯,0.0861667801,-0.133266101,0.158973553,-0.030929314,0.115632488,0.322951751,0.117061066,0.912526483,0.400061044,4-octylphenol
4-octylphenol1,0.1919322615,-0.2855992927,0.4688403629,-0.4613242667,2.3234831092,0.6141764850,1.7872151736,-0.4778250813,0.9492905666,0.0684985358,⋯,0.0464723011,-0.098046006,0.204156632,0.270479648,0.423434660,0.586502351,1.392661378,0.570198885,0.912969638,4-octylphenol
124-Trichlorobenzene0,0.0080013122,0.0048791414,0.0135043987,0.0134059151,0.0004118712,0.0421897391,-0.0186243209,0.0045459801,0.0011581828,0.0820340491,⋯,0.0023898678,-0.003857041,0.005482283,-0.005999139,0.012015410,-0.027034034,0.069573587,0.131426491,-0.007837363,124-Trichlorobenzene
124-Trichlorobenzene1,0.0150202732,0.0208173917,0.0415044430,0.0180178136,0.0556598030,-0.0632748582,0.0416242482,0.0182982485,0.0045270320,-0.1114213676,⋯,0.1671736117,0.030992751,-0.018535870,0.019734837,0.002392184,-0.041270413,0.195565456,-0.090925733,0.209472727,124-Trichlorobenzene
A7360,-0.0083587626,0.0129428213,-0.0122405416,0.0383206750,0.0023603288,0.0014072831,-0.0522923263,-0.0168930498,0.0002765820,0.0489670559,⋯,-0.0015143726,0.002437177,-0.003432376,0.003649616,-0.005972552,0.004682305,0.027621433,-0.020727465,0.008075005,A736


In [36]:
## group substances by median to have only one obs per susbtance (makes some results 'easier' and more interpretable for our task):
behaviour_median <- df_new %>% 
  group_by(full_name) %>% 
  summarise(across(where(is.numeric), \(x) median(x, na.rm = TRUE))) %>% 
  column_to_rownames("full_name")

In [37]:
behaviour_median

,bspline_1_1,bspline_2_1,bspline_3_1,bspline_4_1,bspline_5_1,bspline_6_1,bspline_7_1,bspline_8_1,bspline_9_1,bspline_1_2,⋯,bspline_9_2,bspline_1_3,bspline_2_3,bspline_3_3,bspline_4_3,bspline_5_3,bspline_6_3,bspline_7_3,bspline_8_3,bspline_9_3
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1-2Dichloroethane,0.0428599502,0.3104169809,0.5140330488,0.269480113,0.0432431615,0.1191663208,-0.161778078,0.023124221,-0.3382460723,0.3191718511,⋯,0.046915940,0.1500494456,-0.2395163495,0.3278025205,-0.311698492,-0.767715644,-0.787530033,-0.6808124247,-7.705458e-01,-0.6110399816
1-chlorodecane,-0.0012663269,0.0212297344,-0.0143353314,0.035529260,0.0156677416,0.0116948309,0.060606854,0.070957249,0.0206552934,0.0071250680,⋯,0.022286039,0.0000000000,0.0000000000,0.0000000000,0.000000000,0.000000000,0.000000000,0.0000000000,0.000000e+00,0.0000000000
124-Trichlorobenzene,0.0115107927,0.0128482665,0.0275044208,0.015711864,0.0280358371,-0.0105425596,0.011499964,0.011422114,0.0028426074,-0.0146936592,⋯,0.004155809,0.0847817398,0.0135678547,-0.0065267937,0.006867849,0.007203797,-0.034152224,0.1325695214,2.025038e-02,0.1008176821
4-octylphenol,0.1919322615,-0.2855992927,0.4688403629,-0.461324267,2.3234831092,0.6141764850,1.787215174,-0.477825081,0.9492905666,0.0069378247,⋯,1.463942855,0.0663195406,-0.1156560535,0.1815650921,0.119775167,0.269533574,0.454727051,0.7548612223,7.413627e-01,0.6565153412
A736,0.0001266427,-0.0002040018,0.0002881707,0.001465421,0.0005435396,-0.0005483989,0.001493525,-0.016893050,0.0002838245,0.0489670559,⋯,0.004387048,-0.0015143726,0.0000000000,-0.0034323761,0.000000000,0.000000000,0.000000000,0.0215188361,-1.236712e-02,0.0044809306
Acetone,-0.1037677172,0.2436964285,-0.2286078720,0.459528151,0.2982836627,0.0379303268,0.193872492,-0.035983122,0.0639831126,0.0008141887,⋯,-0.006512883,-0.0158548049,0.0257658226,-0.0374403080,0.043701629,-0.120702916,-0.051736799,-0.0426854873,-1.200992e-01,0.0667071195
Acide Acrylique,0.0027422745,0.0027082810,-0.0120392158,0.017406048,-0.0417221223,0.0012836202,-0.011366781,-0.034010862,-0.0567938863,-0.1856440332,⋯,-0.156064299,-0.0009290340,0.0014979329,-0.0021224431,0.002300190,-0.004326072,0.007937426,-0.0134205484,1.950719e-02,0.0177113287
Anthracene,0.0168038689,0.0203822385,0.0055032569,0.031755517,0.0128406529,0.0517040004,0.011609127,-0.005205035,0.0247493056,0.0315735647,⋯,0.017014074,0.0115970186,-0.0331972565,0.0949231698,-0.022010234,0.016956150,0.034157819,0.0818012303,4.424704e-01,0.3242820970
Benzene,-0.0032100417,0.0051140268,-0.0069609412,0.006579335,0.0009656230,0.0092986515,0.010225273,0.019241221,0.0124475315,0.0546834443,⋯,-0.099576610,0.0078233008,-0.0039688936,-0.0349869721,0.042674736,-0.158554275,0.087981225,-0.0599329844,3.363773e-02,-0.0120739767


In [38]:
write.csv(behaviour_median, "../data/multispec_bsplines_median_agreg2_IGT.csv")